<!-- nav -->
[← 8.05 · TypeScript Patterns](08_05_TypeScript_Patterns.ipynb) · [Home](../../README.md) · [8.07 · C Types →](08_07_C_Types.ipynb)

# 8.06 · TypeScript RegExp

TypeScript writes a regular expression as a literal, between slashes, its flags after:

```text
const repeated = /(?<word>[a-z]+)-\k<word>/giv;
```

ECMA-262 defines what is between the slashes, its pattern grammar (22.2.1, "Patterns"), and the flags. `TypeScript
RegExp` is that grammar's dialect, its kinds regexpp's node types, ECMAScript's reference parser's, lowercased:
`regexpliteral`, `pattern`, `alternative`, `capturinggroup`, `quantifier`, `characterclass` and the rest, with two
helpers, a literal's `Flags` and a group's `ModifierFlags`
([CAP-281: Python's, ECMAScript's and C++'s regular expressions as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-281.md)).
A regular expression literal is TypeScript5's `constant` (tutorial 3, 03.7), which holds its spelling, `pattern` and
`flags`, beside the regular expression they encode, its `regexp`, a term of this dialect.

This page builds regular expressions from a single character up, in the builders' fluent calls, prints each as
ECMAScript writes it, puts one in TypeScript5's literal, validates them, and reads the editions of ECMA-262 that have
its kinds. Like the rest of Part I, it starts from `dialects()`; `regexp` holds a spec in the dialect's holder,
`S.TypeScript.RegExp.Expression`, `char` writes a character and `seq` an alternative of elements. No reader here reads
a regular expression's text, in either language: reading TypeScript is mbse-programs' (tutorial 3).

In [1]:
import assert from "node:assert/strict";
import { Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as TypeScriptRegExp from "@mbse/expressions/TypeScript/RegExp";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const REGEXP = TypeScriptRegExp.Expressions.DIALECT;
const ToText = TypeScriptRegExp.Text.ToText;
const char = (c: string) => (e: any) => e.character((h: any) => h.value(BigInt(c.codePointAt(0)!))); // the character c, its value a code point
const seq = (...elements: any[]) => (e: any) => e.alternative((a: any) => a.elements(elements)); // an alternative of elements

/** The regular expression `spec` writes, held by its dialect's holder, written in a transaction of its own. */
function regexp(spec: any): any {
  return store.transaction(() => S.TypeScript.RegExp.Expression((e: any) => e.root(spec)));
}

/** A regular expression's text, as ECMAScript writes it, and its kind. */
function show(held: any): void {
  console.log(ToText(held).padEnd(32), Terms.kindOf(Terms.term(held))[0]);
}

/** A form's attributes, `name=value` each. */
const attributes = (form: any) => [...form.attributes.entries()].map(([k, v]: [string, unknown]) => `${k}=${v}`).join(", ");

<a id="T08.06.1"></a>
## 08.06.1 · A character, a pattern, a literal

The smallest regular expression is one character, `a`, its `value` a code point. Characters in a row are an
`alternative`; alternatives joined by `|` a `pattern`; and a pattern between slashes, with its flags, a `regexpliteral`,
the flags a `Flags` helper of ECMAScript's eight, `d`, `g`, `i`, `m`, `s`, `u`, `v` and `y`, each a boolean named as
regexpp names it (`isGlobal` for `g`, `ignoreCase` for `i`):

In [2]:
const pattern = (e: any) => e.pattern((p: any) => p.alternatives([seq(char("a"), char("b")), seq(char("c"))])); // ab|c
for (const spec of [char("a"), seq(char("a"), char("b")), pattern,
  (r: any) => r.regexpliteral((l: any) => l.pattern(pattern).flags((f: any) => f.isGlobal(true).ignoreCase(true)))]) {
  show(regexp(spec));
}

a                                character


ab                               alternative


ab|c                             pattern


/ab|c/gi                         regexpliteral


<a id="T08.06.2"></a>
## 08.06.2 · Classes, sets and quantifiers

A **character class**, `[a-z_]`, matches one of its elements, characters and ranges, `[^...]` none of them. A
**character set** is a class ECMAScript names: `.`, `\d`, `\s`, `\w` and their negations, each a `characterset` of its
`setKind`, and a Unicode property, `\p{Script=Greek}`, its `key` and `value`. A **quantifier** repeats its `element`
between its `min` and its `max`; the printer writes its shortest spelling, `*`, `+`, `?` or `{2,5}`, then `?` where it is
lazy:

In [3]:
const lowercase = (e: any) => e.characterclassrange((r: any) => r.min(char("a")).max(char("z")));
const sets = [
  (r: any) => r.characterclass((c: any) => c.elements([lowercase, char("_")])),
  (r: any) => r.characterclass((c: any) => c.negate(true).elements([lowercase])),
  (r: any) => r.characterset((s: any) => s.setKind("any")),
  (r: any) => r.characterset((s: any) => s.setKind("digit")),
  (r: any) => r.characterset((s: any) => s.setKind("word").negate(true)),
  (r: any) => r.characterset((s: any) => s.setKind("property").key("Script").value("Greek")),
  (r: any) => r.quantifier((q: any) => q.min(0n).element(char("a"))),
  (r: any) => r.quantifier((q: any) => q.min(1n).greedy(false).element((e: any) => e.characterset((s: any) => s.setKind("digit")))),
  (r: any) => r.quantifier((q: any) => q.min(2n).max(5n).element(char("a"))),
];
for (const spec of sets) show(regexp(spec));

[a-z_]                           characterclass


[^a-z]                           characterclass


.                                characterset


\d                               characterset


\W                               characterset


\p{Script=Greek}                 characterset


a*                               quantifier


\d+?                             quantifier


a{2,5}                           quantifier


<a id="T08.06.3"></a>
## 08.06.3 · Groups, assertions, references and modifiers

A **group**, `(?:a|b)`, gathers alternatives; a **capturing group**, `(a)` or `(?<word>a)`, numbered as it opens and
perhaps named; a **backreference**, `\1` or `\k<word>`, matches what one matched, its `ref` a number or a name. An
**assertion** matches a place: `^`, `$`, `\b`, and the lookarounds, `(?=a)` and `(?<!a)`, which hold alternatives. Two
editions added more: the `v` flag's class expressions (ES2024), `[\w--a]` a word character but `a`, an
`expressioncharacterclass` of a `classsubtraction`; and **modifiers** (ES2025), `(?i:a)`, a group's `modifiers`, which
add or remove flags within it. The opening example, whole:

In [4]:
const word = (e: any) => e.capturinggroup((g: any) => g.name("word").alternatives([seq(
  (e: any) => e.quantifier((q: any) => q.min(1n).element((c: any) => c.characterclass((k: any) => k.elements([lowercase])))))]));
const repeated = regexp((r: any) => r.regexpliteral((l: any) => l
  .pattern((e: any) => e.pattern((p: any) => p.alternatives([seq(word, char("-"), (e: any) => e.backreference((b: any) => b.ref("word")))])))
  .flags((f: any) => f.isGlobal(true).ignoreCase(true).unicodeSets(true))));
show(repeated);
const groups = [
  (r: any) => r.group((g: any) => g.alternatives([seq(char("a")), seq(char("b"))])),
  seq((e: any) => e.capturinggroup((g: any) => g.alternatives([seq(char("a"))])), (e: any) => e.backreference((b: any) => b.ref(1n))),
  seq((e: any) => e.assertion((a: any) => a.assertionKind("start")), char("a"),
    (e: any) => e.assertion((a: any) => a.assertionKind("word")),
    (e: any) => e.assertion((a: any) => a.assertionKind("lookahead").alternatives([seq(char("b"))])),
    (e: any) => e.assertion((a: any) => a.assertionKind("lookbehind").negate(true).alternatives([seq(char("c"))])),
    (e: any) => e.assertion((a: any) => a.assertionKind("end"))),
  (r: any) => r.expressioncharacterclass((x: any) => x.expression((e: any) => e.classsubtraction((s: any) => s
    .left((w: any) => w.characterset((t: any) => t.setKind("word"))).right(char("a"))))),
  (r: any) => r.group((g: any) => g.modifiers((m: any) => m.modifiers((s: any) => s.add((a: any) => a.ignoreCase(true))))
    .alternatives([seq(char("a"))])),
];
for (const spec of groups) show(regexp(spec));

/(?<word>[a-z]+)-\k<word>/giv    regexpliteral


(?:a|b)                          group


(a)\1                            alternative


^a\b(?=b)(?<!c)$                 alternative


[\w--a]                          expressioncharacterclass


(?i:a)                           group


<a id="T08.06.4"></a>
## 08.06.4 · A spelling kept beside what it encodes

A reader keeps how a regular expression was written where the tree does not give it: a character's `raw`, `\x41` for
65, a property's, `\p{L}` for the general category `L`, a quantifier's, a backreference's and the flags'. The printer
writes a `raw` back as written, and, where there is none, the default spelling: a character escaped where it is
syntax, `\.`, a property by its key and value
([PRI-023: domain representations](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-023)):

In [5]:
const letter = (s: any) => s.setKind("property").key("General_Category").value("L");
const spellings = [
  (r: any) => r.character((c: any) => c.raw("\\x41").value(65n)), (r: any) => r.character((c: any) => c.value(65n)),
  char("."),
  (r: any) => r.characterset((s: any) => letter(s.raw("\\p{L}"))), (r: any) => r.characterset(letter),
  (r: any) => r.quantifier((q: any) => q.raw("a{0,}").min(0n).element(char("a"))),
];
for (const spec of spellings) {
  const held = regexp(spec);
  console.log(ToText(held).padEnd(32), Terms.kindOf(Terms.term(held))[0].padEnd(13), attributes(Terms.form(Terms.term(held))));
}

\x41                             character     raw=\x41, value=65


A                                character     value=65


\.                               character     value=46


\p{L}                            characterset  raw=\p{L}, setKind=property, key=General_Category, value=L


\p{General_Category=L}           characterset  setKind=property, key=General_Category, value=L


a{0,}                            quantifier    raw=a{0,}, min=0


<a id="T08.06.5"></a>
## 08.06.5 · In TypeScript5's literal

TypeScript5's `constant` is a regular expression literal: its `pattern` and `flags` the spelling a reader read, and its
`regexp` the regular expression they encode, a subterm of `TypeScript RegExp`' `Term`, the one property of the family
that holds one. A constant written from code may hold its `regexp` alone, and the printer writes it from that.
TypeScript5 validates the constant with its regular expression, infers its domain, `RegExp`, and, as it folds its literals alone, leaves it in the residual, named. A string given to `new RegExp("a+")` is a string, not this dialect's term: no reader guesses
it is one ([PRI-025: never guess the schema](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-025)):

In [6]:
const literal = (r: any) => r.regexpliteral((l: any) => l.pattern((e: any) => e.pattern((p: any) => p.alternatives([seq(
  (e: any) => e.quantifier((q: any) => q.min(1n).element(char("a"))))]))).flags((f: any) => f.raw("g").isGlobal(true))); // /a+/g
const [spelled, written, called]: any[] = store.transaction(() => [
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.constant((c: any) => c.pattern("a+").flags("g").regexp(literal)))),
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.constant((c: any) => c.regexp(literal)))),
  S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.newexpression((n: any) => n
    .callee((c: any) => c.identifier((i: any) => i.name("RegExp")))
    .arguments([(a: any) => a.literal((l: any) => l.value((v: any) => v.as_scalar("a+")).quote('"'))])))),
]);
const dialect = TypeScript5.Expressions.DIALECT;
for (const held of [spelled, written]) {
  const form = Terms.form(Terms.term(held));
  console.log(TypeScript5.Text.ToText(held).padEnd(8), form.kind, attributes(form) || "-", "| regexp:", Terms.kindOf(form.arguments[0])[0],
    "| problems:", dialect.validate(held).join("; ") || "none", "| domain:", dialect.infer(held).name());
}
try {
  TypeScript5.Simplifiers.OfAny(spelled).evaluate();
  throw new Error("evaluated");
} catch (error) {
  assert.equal((error as Error).name, "NotEvaluated");
  console.log((error as Error).message);
}
console.log(TypeScript5.Text.ToText(called), "| its terms' dialects:",
  [...new Set([...Terms.walk(Terms.term(called))].map((t: any) => Terms.dialectOf(t)))].sort().join(", "));

/a+/g    constant pattern=a+, flags=g | regexp: regexpliteral | problems: none | domain: RegExp


/a+/g    constant - | regexp: regexpliteral | problems: none | domain: RegExp


the root is not a literal: TypeScript5 constant is not simplified


new RegExp("a+") | its terms' dialects: Expressions.TypeScript.TypeScript5.Term


<a id="T08.06.6"></a>
## 08.06.6 · Validated, binding nothing, not simplified

Validation reports ECMA-262's early errors that a term's data can show, in its own words: a quantifier's bounds out of
order, flags given twice, both `u` and `v`, a modifier both added and removed, a range out of order. A named group's
name is a label its backreferences resolve, not a name the expression binds: a regular expression needs nothing from a
scope. And it is matched by the engine, not simplified: TypeScript5's rules fold none of its kinds: each stays in the residual, named by its dialect:

In [7]:
const refused = [
  (r: any) => r.quantifier((q: any) => q.min(2n).max(1n).element(char("a"))),
  (r: any) => r.regexpliteral((l: any) => l.pattern((e: any) => e.pattern((p: any) => p.alternatives([seq(char("a"))])))
    .flags((f: any) => f.raw("gg").isGlobal(true))),
  (r: any) => r.regexpliteral((l: any) => l.pattern((e: any) => e.pattern((p: any) => p.alternatives([seq(char("a"))])))
    .flags((f: any) => f.unicode(true).unicodeSets(true))),
  (r: any) => r.characterclassrange((c: any) => c.min(char("z")).max(char("a"))),
];
for (const spec of refused) {
  const held = regexp(spec);
  console.log(ToText(held).padEnd(10), REGEXP.validate(held).join("; "));
}
console.log(ToText(repeated), "| free:", [...Symbolics.free(repeated, REGEXP)].join(", ") || "none", "| problems:",
  REGEXP.validate(repeated).join("; ") || "none");
try {
  TypeScript5.Simplifiers.OfAny(repeated).evaluate();
  throw new Error("evaluated");
} catch (error) {
  assert.equal((error as Error).name, "NotEvaluated");
  console.log((error as Error).message);
}

a{2,1}     a quantifier's max 1 is less than its min 2


/a/gg      a regular expression's flags cannot be 'gg'


/a/uv      a regular expression's flags cannot have both u and v


z-a        a characterclassrange's min 122 is greater than its max 97


/(?<word>[a-z]+)-\k<word>/giv | free: none | problems: none


the root is not a literal: TypeScript RegExp regexpliteral is not simplified


<a id="T08.06.7"></a>
## 08.06.7 · Its editions as data

`TypeScript RegExp` covers ECMA-262's editions from the 5.1 Edition to the 17th, each found by Ecma International's
designation or a common name, `ES2015` or `ECMAScript 2024`
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)),
and records which of its kinds each has: the `v` flag's five from the 15th Edition (ES2024), `modifiers` from the 16th
(ES2025); the 17th's pattern grammar is the 16th's:

In [8]:
const regexpDialect = store.Instances.Expressions.TypeScript.RegExp;
const editions = [...Standards.standards(regexpDialect).keys()];
console.log(editions.length, "editions:", editions[0], "to", editions.at(-1));
for (const tag of ["ES5.1", "ES2023", "ES2024", "ES2025", "ES2026"]) {
  const edition = Standards.find(regexpDialect, tag)!;
  console.log(tag.padEnd(7), Standards.tags(edition)[0]!.padEnd(21), Standards.kinds(edition).length, "kinds");
}
const modifiers = store.registered("Expressions.TypeScript.RegExp.Modifiers");
console.log("Modifiers since", Standards.tags([...Standards.of(regexpDialect, modifiers).values()][0]!).at(-1));

13 editions: ECMA-262 5.1 Edition to ECMA-262 17th Edition


ES5.1   ECMA-262 5.1 Edition  12 kinds


ES2023  ECMA-262 14th Edition 12 kinds


ES2024  ECMA-262 15th Edition 17 kinds


ES2025  ECMA-262 16th Edition 18 kinds


ES2026  ECMA-262 17th Edition 18 kinds


Modifiers since ES2025


## What you learned

- **A regular expression is a term of its own dialect**, `TypeScript RegExp`, regexpp's node types as kinds, from a
  character up to a literal with its flags.
- **TypeScript5's literal holds one** as its `regexp`, beside its spelling, the one place of the family that does; a
  string given to `new RegExp` stays a string.
- **A spelling is kept beside what it encodes**, and the printer writes it back, or a default spelling.
- **Validation reports ECMA-262's early errors**; a named group's name is a label, so it binds nothing; simplification leaves each kind in the residual, named.
- **Its editions are data**: ECMA-262's 5.1 Edition to the 17th, the `v` flag's classes and modifiers where they began.

That ends the Python and TypeScript families' sublanguages. Next, Part II puts expressions to work as constraints,
starting with case study 9.

<!-- nav -->
[← 8.05 · TypeScript Patterns](08_05_TypeScript_Patterns.ipynb) · [Home](../../README.md) · [8.07 · C Types →](08_07_C_Types.ipynb)